# Lesson 05 — Advanced NLP

Covers: tokenisation internals (BPE), sentence embeddings, semantic search with FAISS, RAG basics, NER/POS overview.

## 1. Tokenisation — BPE from Scratch

In [ ]:
from collections import defaultdict, Counter

def get_vocab(corpus):
    """Build initial character-level vocabulary with end-of-word marker."""
    vocab = defaultdict(int)
    for word, freq in corpus.items():
        chars = " ".join(list(word)) + " </w>"
        vocab[chars] += freq
    return vocab

def get_pairs(vocab):
    """Count all adjacent symbol pairs across vocabulary."""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge most frequent pair across all vocab entries."""
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

# Toy corpus
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3, "new": 8}
vocab = get_vocab(corpus)
print("Initial vocab:")
for k, v in list(vocab.items())[:3]:
    print(f"  '{k}': {v}")

# Run 10 BPE merge operations
for i in range(10):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge {i+1}: {best} (freq={pairs[best]})")


## 2. Why Tokenisation Matters for Performance

| Tokeniser | Vocab Size | OOV Handling | Notes |
|---|---|---|---|
| Word-level | 50k-100k | Poor (UNK) | Simple but breaks on rare words |
| Char-level | ~100 | Perfect | Long sequences, slow |
| BPE | 32k-50k | Good (subwords) | GPT-2, RoBERTa |
| WordPiece | 30k | Good | BERT (uses `##` prefix) |
| Unigram LM | 32k | Good | SentencePiece / T5 |

**Interview insight:** BPE is greedy (deterministic merges). WordPiece maximises likelihood of training data. SentencePiece works on raw text without pre-tokenisation — language-agnostic.

## 3. Sentence Embeddings with Hugging Face

In [ ]:
# pip install sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")  # 22M params, fast

sentences = [
    "The cat sat on the mat.",
    "A feline rested on a rug.",
    "Deep learning requires large datasets.",
    "Neural networks need lots of training data.",
    "Python is a popular programming language.",
]

embeddings = model.encode(sentences, normalize_embeddings=True)
print(f"Embeddings shape: {embeddings.shape}")  # (5, 384)

# Cosine similarity (dot product since normalised)
sim_matrix = embeddings @ embeddings.T
print("\nCosine similarity matrix:")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s[:40]}")
print()
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        print(f"  sim({i},{j}) = {sim_matrix[i,j]:.3f}")


## 4. Semantic Search with FAISS

In [ ]:
# pip install faiss-cpu

import faiss
import numpy as np

# Build an index over a corpus
documents = [
    "How to train a neural network",
    "Backpropagation algorithm explained",
    "Python list comprehensions tutorial",
    "Fine-tuning BERT for classification",
    "SQL joins and database normalisation",
    "Transfer learning in computer vision",
    "Gradient descent optimisation methods",
    "Docker containers for ML deployment",
]

# In practice: encode with SentenceTransformer
# Here: use random vectors as proxy
np.random.seed(42)
dim = 384
corpus_vecs = np.random.randn(len(documents), dim).astype("float32")
faiss.normalize_L2(corpus_vecs)

# Build flat (exact) index — for production use IVFFlat or HNSW
index = faiss.IndexFlatIP(dim)   # Inner Product (= cosine after normalisation)
index.add(corpus_vecs)
print(f"Index contains {index.ntotal} vectors")

# Query
query_vec = np.random.randn(1, dim).astype("float32")
faiss.normalize_L2(query_vec)
scores, indices = index.search(query_vec, k=3)

print("\nTop-3 results:")
for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
    print(f"  {rank+1}. [{score:.3f}] {documents[idx]}")


## 5. RAG — Retrieval-Augmented Generation Pipeline

In [ ]:
# RAG architecture (conceptual — requires API keys for LLM call):
#
#   User query
#       |
#       v
#   [Embedding model] --> query vector
#       |
#       v
#   [Vector store] --> top-K relevant chunks
#       |
#       v
#   [Prompt assembly] --> "Context: {chunks}\nQuestion: {query}"
#       |
#       v
#   [LLM] --> answer grounded in retrieved context

def build_rag_prompt(query, retrieved_chunks, k=3):
    context = "\n\n".join("[{}] {}".format(i+1, chunk) for i, chunk in enumerate(retrieved_chunks[:k]))
    template = (
        "Answer the question using ONLY the context below. "
        'If the answer is not in the context, say \"I don\'t know.\"\n\n'
        "Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    )
    return template.format(context=context, query=query)

chunks = [
    "LoRA reduces trainable parameters by decomposing weight updates into low-rank matrices.",
    "FAISS is a library for efficient similarity search over dense vectors.",
    "Mixed precision training uses fp16 for forward passes to reduce memory usage.",
]
prompt = build_rag_prompt("How does LoRA work?", chunks)
print(prompt)


## 6. NER / POS — Trade-off Comparison

| Approach | Pros | Cons | When to use |
|---|---|---|---|
| Rule-based (regex) | Fast, interpretable | Brittle, no generalisation | Known patterns, phone numbers |
| CRF | Good sequence modelling | Needs hand-crafted features | Small data, needs structure |
| BiLSTM-CRF | Contextual, end-to-end | Slower, needs tuning | Mid-size corpora |
| BERT fine-tune | SOTA accuracy | GPU required, slow inference | Quality-critical tasks |
| spaCy (pretrained) | Fast, production-ready | Less customisable | Rapid prototyping |

**Q: Why does BERT outperform BiLSTM-CRF on NER?**  
BERT's bidirectional pretraining captures long-range context and sub-word semantics. `##Smith` in `Blacksmith` vs. `John Smith` gets different representations. BiLSTM must learn this from scratch with far less data.

## Interview Q&A

**Q: What's the difference between BPE and WordPiece?**  
BPE merges the most frequent pair iteratively (greedy count-based). WordPiece merges the pair that maximises the language model likelihood of the training data — statistically principled but slower. GPT uses BPE, BERT uses WordPiece.

**Q: When would you use FAISS IVFFlat over IndexFlatIP?**  
FlatIP is exact but O(n) per query. IVFFlat clusters vectors into Voronoi cells and only searches nearby cells — O(√n) approximate. Use IVFFlat when corpus > ~100k vectors and latency matters more than perfect recall.

**Q: RAG vs. fine-tuning — which to choose?**  
RAG when: knowledge changes frequently, you need citations/sourcing, data is private. Fine-tuning when: you need the model to adopt a style/persona, task is highly domain-specific, latency budget is tight (no retrieval step).